In [1]:
import torch.nn as nn
import torch.functional as F
from typing import Any
import torchview
from torch import Tensor
import torch

In [2]:
from blocks.ddpm_blocks import UpBlockUnet, MidBlock, DownBlock

In [3]:
import numpy as np
from tqdm import tqdm
from torch.optim import Adam
from torch.utils.data import DataLoader

from models.vqvae import VQVAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

########################
# Replace config dicts with variables
num_timesteps = 100
beta_start = 0.0001
beta_end = 0.02

dataset_name = "mnist"
im_path = "mnist/"
im_size = 32
im_channels = 1

# Must match VQVAE.z_channels (the VAE encodes to a 3-channel latent)
z_channels = 3

EPOCHS = 10
IMAGE_CHANNELS = 1

task_name = "vqvae_train"
vqvae_latent_dir_name = 0
ldm_batch_size = 0
vqvae_autoencoder_ckpt_name = "vqvae"
ldm_epochs = 10
ldm_lr = 2e-4
ldm_ckpt_name = "ddpm_class_cond"

# Condition config variables
condition_types = ["class"]
num_classes = 10
text_embed_dim = 256
image_condition_input_channels = 1
image_condition_output_channels = 1
class_cond_drop_prob = 0.0
use_latents = True


# Instantiate Condition related components
text_tokenizer = None
text_model = None
empty_text_embed = None

# Support for easily checking if we should use condition
use_condition = condition_types is not None and len(condition_types) > 0


In [4]:
import glob
import os
import torchvision
from PIL import Image
from tqdm import tqdm
from torch.utils.data.dataset import Dataset
import pickle

def load_latents(latent_path):
    r"""
    Simple utility to save latents to speed up ldm training
    :param latent_path:
    :return:
    """
    latent_maps = {}
    for fname in glob.glob(os.path.join(latent_path, "*.pkl")):
        s = pickle.load(open(fname, "rb"))
        for k, v in s.items():
            latent_maps[k] = v[0]
    return latent_maps

class MnistDataset(Dataset):
    r"""
    Nothing special here. Just a simple dataset class for mnist images.
    Created a dataset class rather using torchvision to allow
    replacement with any other image dataset
    """

    def __init__(
        self,
        split,
        im_path,
        im_size,
        im_channels,
        use_latents=False,
        latent_path=None,
        condition_config=None,
    ):
        r"""
        Init method for initializing the dataset properties
        :param split: train/test to locate the image files
        :param im_path: root folder of images
        :param im_ext: image extension. assumes all
        images would be this type.
        """
        self.split = split
        self.im_size = im_size
        self.im_channels = im_channels

        # Should we use latents or not
        self.latent_maps = None
        self.use_latents = False

        # Conditioning for the dataset
        self.condition_types = (
            [] if condition_config is None else condition_config["condition_types"]
        )

        self.images, self.labels = self.load_images(im_path)

        # Whether to load images and call vae or to load latents
        if use_latents and latent_path is not None:
            latent_maps = load_latents(latent_path)
            if len(latent_maps) == len(self.images):
                self.use_latents = True
                self.latent_maps = latent_maps
                print("Found {} latents".format(len(self.latent_maps)))
            else:
                print("Latents not found")

    def load_images(self, im_path):
        r"""
        Gets all images from the path specified
        and stacks them all up
        :param im_path:
        :return:
        """
        assert os.path.exists(im_path), "images path {} does not exist".format(im_path)
        ims = []
        labels = []
        for d_name in tqdm(os.listdir(im_path)):
            fnames = glob.glob(os.path.join(im_path, d_name, "*.{}".format("png")))
            fnames += glob.glob(os.path.join(im_path, d_name, "*.{}".format("jpg")))
            fnames += glob.glob(os.path.join(im_path, d_name, "*.{}".format("jpeg")))
            for fname in fnames:
                ims.append(fname)
                if "class" in self.condition_types:
                    labels.append(int(d_name))
        print("Found {} images for split {}".format(len(ims), self.split))
        return ims, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        ######## Set Conditioning Info ########
        cond_inputs = {}
        if "class" in self.condition_types:
            cond_inputs["class"] = self.labels[index]
        #######################################

        if self.use_latents:
            latent = self.latent_maps[self.images[index]]
            if len(self.condition_types) == 0:
                return latent
            else:
                return latent, cond_inputs
        else:
            im = Image.open(self.images[index])
            im = im.convert("L" if self.im_channels == 1 else "RGB")
            im = im.resize((self.im_size, self.im_size))    
            im_tensor = torchvision.transforms.ToTensor()(im)

            # Convert input to -1 to 1 range.
            im_tensor = (2 * im_tensor) - 1
            if len(self.condition_types) == 0:
                return im_tensor
            else:
                return im_tensor, cond_inputs


import torchvision
import os

from torchvision.transforms import ToTensor

mn = torchvision.datasets.MNIST("mnist", download=True, transform=ToTensor())
labels = mn.train_labels
data = mn.train_data
for idx, label in enumerate(mn.train_labels):
    img = data[idx]
    if not os.path.exists(f"mnist/{label}"):
        os.mkdir(f"mnist/{label}")

    fpath = f"mnist/{label}/{idx}.png"
    if not os.path.exists(fpath):
        torchvision.utils.save_image(img.float() / 255.0, fpath)


dataset = MnistDataset(split="train", im_path="mnist", im_size=32, im_channels=1, condition_config = {
            "condition_types": ["class"],
            "class_condition_config": {
                "num_classes": 10,
            },
            "text_condition_config": {
                "text_embed_dim": 256,
            },
            "image_condition_config": {
                "image_condition_input_channels": 1,
                "image_condition_output_channels": 1,
            },
        }, use_latents=use_latents, latent_path = os.path.join(task_name, vqvae_autoencoder_ckpt_name)
)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

/home/aslonhamidov/.conda/envs/py311_gcc13/lib/python3.11/site-packages/torchvision/datasets/mnist.py:66: UserWarning: train_labels has been renamed targets
  warnings.warn("train_labels has been renamed targets")
/home/aslonhamidov/.conda/envs/py311_gcc13/lib/python3.11/site-packages/torchvision/datasets/mnist.py:76: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")
100%|██████████| 11/11 [00:00<00:00, 123.05it/s]

Found 60000 images for split train
Latents not found


In [5]:
next(iter(loader))[0].shape, next(iter(loader))[1]

(torch.Size([64, 1, 32, 32]),
 {'class': tensor([3, 5, 2, 1, 7, 5, 4, 4, 0, 0, 3, 7, 7, 5, 4, 6, 9, 4, 9, 8, 1, 5, 3, 6,
          9, 7, 5, 5, 7, 6, 8, 5, 2, 7, 3, 7, 6, 0, 5, 6, 2, 7, 0, 4, 3, 0, 8, 2,
          2, 0, 3, 4, 0, 5, 4, 1, 5, 9, 8, 0, 5, 8, 9, 8])})

In [1]:
from torch import einsum


def get_time_embedding(t: Tensor, t_emb_dim):
    factor = 10000 ** (
        (
            torch.arange(
                start=0, end=t_emb_dim // 2, dtype=torch.float32, device=t.device
            )
        )
        / (t_emb_dim // 2)
    )

    t_emb = t[:, None].repeat(1, t_emb_dim // 2) / factor
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=1)

    return t_emb
class UNetBase(nn.Module):
    def __init__(self, im_channels: int, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.down_channels = [64, 128, 256, 256]
        self.mid_channels = [256, 256]
        self.t_emb_dim = 256
        self.down_sample = [True, True, True]
        self.num_down_layers = 2
        self.num_mid_layers = 2
        self.num_up_layers = 2
        self.norm_channels = 32
        self.attns = [True, True, True]
        self.num_heads = 16
        self.conv_out_channels = 1

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
        )

        self.conv_in = nn.Conv2d(
            im_channels, self.down_channels[0], kernel_size=3, padding=1
        )
        self.downs = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            self.downs.append(
                DownBlock(
                    self.down_channels[i],
                    self.down_channels[i + 1],
                    self.t_emb_dim,
                    down_sample=self.down_sample[i],
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    attn=self.attns[i],
                    norm_channels=self.norm_channels,
                )
            )

        self.mids = nn.ModuleList()
        for i in range(len(self.mid_channels) - 1):
            self.mids.append(
                MidBlock(
                    self.mid_channels[i],
                    self.mid_channels[i + 1],
                    self.t_emb_dim,
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    norm_channels=self.norm_channels,
                )
            )

        self.ups = nn.ModuleList()
        for i in reversed(range(len(self.down_channels) - 1)):
            # print(
            #     "In Channels", self.down_channels[i] * 2
            # )
            # print("Out Channels", self.down_channels[i - 1] if i != 0 else self.down_channels[0])
            self.ups.append(
                UpBlockUnet(
                    self.down_channels[i] * 2,
                    self.down_channels[i - 1] if i != 0 else self.down_channels[0],
                    up_sample=self.down_sample[i],
                    t_emb_dim=self.t_emb_dim,
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    norm_channels=self.norm_channels,
                )
            )

        self.norm_out = nn.GroupNorm(
            self.norm_channels,
            self.down_channels[0],  # was self.conv_out_channels
        )
        self.conv_out = nn.Conv2d(
            self.down_channels[0],
            self.conv_out_channels,
            kernel_size=3,
            padding=1,
        )

    def forward(self, x: Tensor, t):
        # shapes assuiming dowblocks - [c1, c2, c3,c4 ...]
        # shapes midblocks - [c4, c4, c3]
        # up samples - [c3, c2, c1]
        # B X C X H X W
        out = self.conv_in(x)
        # B X C1 X H X W
        t_emb = get_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
        t_emb = self.t_proj(t_emb)

        down_outs = []
        for idx, down in enumerate(self.downs):
            down_outs.append(out)
            out = down(out, t_emb)

        for mid in self.mids:
            out = mid(out, t_emb)

        self.up_outs = []
        for idx, up in enumerate(self.ups):
            down_out = down_outs.pop()
            out = up(out, down_out, t_emb)

        out = self.norm_out(out)
        out = nn.SiLU()(out)
        out = self.conv_out(out)

        return out

NameError: name 'Tensor' is not defined

In [7]:
class UnetCond(nn.Module):
    def __init__(self, im_channels: int = 3, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.down_channels = [64, 128, 256, 256]
        self.mid_channels = [256, 256]
        self.t_emb_dim = 256
        # No spatial down/up-sampling: the VAE latent is already tiny (4x4),
        # so further halving would collapse it to a zero-sized feature map.
        self.down_sample = [False, False, False]
        self.num_down_layers = 2
        self.num_mid_layers = 2
        self.num_up_layers = 2
        self.norm_channels = 32
        self.attns = [True, True, True]
        self.num_heads = 16
        # Predict noise in the latent space -> output channels must equal input (z) channels.
        self.conv_out_channels = im_channels

        ######## Class, Mask and Text Conditioning Config #####
        self.class_cond = False
        self.text_cond = False
        self.image_cond = False
        self.text_embed_dim = None
        self.condition_config = {
            "condition_types": ["class"],
            "class_condition_config": {
                "num_classes": 10,
            },
            "text_condition_config": {
                "text_embed_dim": 256,
            },
            "image_condition_config": {
                "image_condition_input_channels": 1,
                "image_condition_output_channels": 1,
            },
        }
        condition_types = self.condition_config["condition_types"]  # ["class", "text", "image"]
        if self.condition_config is not None:
            assert "condition_types" in self.condition_config, (
                "Condition Type not provided in model config"
            )
            if "class" in condition_types:
                # validate_class_config(self.condition_config)
                self.class_cond = True
                self.num_classes = self.condition_config["class_condition_config"][
                    "num_classes"
                ]
            if "text" in condition_types:
                self.text_cond = True
                self.text_embed_dim = self.condition_config["text_condition_config"][
                    "text_embed_dim"
                ]
            if "image" in condition_types:
                self.image_cond = True
                self.im_cond_input_ch = self.condition_config["image_condition_config"][
                    "image_condition_input_channels"
                ]
                self.im_cond_output_ch = self.condition_config[
                    "image_condition_config"
                ]["image_condition_output_channels"]
        if self.class_cond:
            # Rather than using a special null class we dont add the
            # class embedding information for unconditional generation
            self.class_emb = nn.Embedding(self.num_classes, self.t_emb_dim)

        if self.image_cond:
            # Map the mask image to a N channel image and
            # concat that with input across channel dimension
            self.cond_conv_in = nn.Conv2d(
                in_channels=self.im_cond_input_ch,
                out_channels=self.im_cond_output_ch,
                kernel_size=1,
                bias=False,
            )
            self.conv_in_concat = nn.Conv2d(
                im_channels + self.im_cond_output_ch,
                self.down_channels[0],
                kernel_size=3,
                padding=1,
            )
        else:
            self.conv_in = nn.Conv2d(
                im_channels, self.down_channels[0], kernel_size=3, padding=1
            )
        self.cond = self.text_cond or self.image_cond or self.class_cond
        ###################################

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
        )

        self.downs = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            self.downs.append(
                DownBlock(
                    self.down_channels[i],
                    self.down_channels[i + 1],
                    self.t_emb_dim,
                    down_sample=self.down_sample[i],
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    attn=self.attns[i],
                    norm_channels=self.norm_channels,
                )
            )

        self.mids = nn.ModuleList()
        for i in range(len(self.mid_channels) - 1):
            self.mids.append(
                MidBlock(
                    self.mid_channels[i],
                    self.mid_channels[i + 1],
                    self.t_emb_dim,
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    norm_channels=self.norm_channels,
                )
            )

        self.ups = nn.ModuleList()
        for i in reversed(range(len(self.down_channels) - 1)):
            # print(
            #     "In Channels", self.down_channels[i] * 2
            # )
            # print("Out Channels", self.down_channels[i - 1] if i != 0 else self.down_channels[0])
            self.ups.append(
                UpBlockUnet(
                    self.down_channels[i] * 2,
                    self.down_channels[i - 1] if i != 0 else self.down_channels[0],
                    up_sample=self.down_sample[i],
                    t_emb_dim=self.t_emb_dim,
                    num_heads=self.num_heads,
                    num_layers=self.num_down_layers,
                    norm_channels=self.norm_channels,
                )
            )

        self.norm_out = nn.GroupNorm(
            self.norm_channels,
            self.down_channels[0],  # was self.conv_out_channels
        )
        self.conv_out = nn.Conv2d(
            self.down_channels[0],
            self.conv_out_channels,
            kernel_size=3,
            padding=1,
        )
        
        
    def forward(self, x, t, cond_input = None):
         # shapes assuiming dowblocks - [c1, c2, c3,c4 ...]
        # shapes midblocks - [c4, c4, c3]
        # up samples - [c3, c2, c1]
        # B X C X H X W
        out = self.conv_in(x)
        # B X C1 X H X W
        t_emb = get_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
        t_emb = self.t_proj(t_emb)
        
        
        
        #### conditioning part
        if self.class_cond:
            assert cond_input is not None and "class" in cond_input, (
                "Model initialized with class conditioning but cond_input has no class"
            )
            # one-hot (B, num_classes) @ class embedding (num_classes, t_emb_dim) -> (B, t_emb_dim)
            class_embed = cond_input["class"].float() @ self.class_emb.weight
            t_emb = t_emb + class_embed
        #### conditioning part

        down_outs = []
        for idx, down in enumerate(self.downs):
            down_outs.append(out)
            out = down(out, t_emb)

        for mid in self.mids:
            out = mid(out, t_emb)

        self.up_outs = []
        for idx, up in enumerate(self.ups):
            down_out = down_outs.pop()
            out = up(out, down_out, t_emb)

        out = self.norm_out(out)
        out = nn.SiLU()(out)
        out = self.conv_out(out)
        return out


# Smoke test: one-hot the class labels and match the timestep batch to the image batch.
# (This runs the conditioning path on raw loader images; with down_sample all False the
# spatial size is preserved, so output shape == input shape.)
_imgs, _cond = next(iter(loader))
_cond_oh = {"class": torch.nn.functional.one_hot(_cond["class"], num_classes).float()}
# UnetCond(im_channels=1)(_imgs, torch.randint(0, num_timesteps, (_imgs.shape[0],)), _cond_oh).shape

In [8]:
import torch
import numpy as np


class LinearNoiseScheduler:
    r"""
    Class for the linear noise scheduler that is used in DDPM.
    """
    
    def __init__(self, num_timesteps, beta_start, beta_end):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end
        # Mimicking how compvis repo creates schedule
        self.betas = (
                torch.linspace(beta_start ** 0.5, beta_end ** 0.5, num_timesteps) ** 2
        )
        self.alphas = 1. - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1 - self.alpha_cum_prod)
    
    def add_noise(self, original, noise, t):
        r"""
        Forward method for diffusion
        :param original: Image on which noise is to be applied
        :param noise: Random Noise Tensor (from normal dist)
        :param t: timestep of the forward process of shape -> (B,)
        :return:
        """
        original_shape = original.shape
        batch_size = original_shape[0]
        
        sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        
        # Reshape till (B,) becomes (B,1,1,1) if image is (B,C,H,W)
        for _ in range(len(original_shape) - 1):
            sqrt_alpha_cum_prod = sqrt_alpha_cum_prod.unsqueeze(-1)
        for _ in range(len(original_shape) - 1):
            sqrt_one_minus_alpha_cum_prod = sqrt_one_minus_alpha_cum_prod.unsqueeze(-1)
        
        # Apply and Return Forward process equation
        return (sqrt_alpha_cum_prod.to(original.device) * original
                + sqrt_one_minus_alpha_cum_prod.to(original.device) * noise)
    
    def sample_prev_timestep(self, xt, noise_pred, t):
        r"""
            Use the noise prediction by model to get
            xt-1 using xt and the nosie predicted
        :param xt: current timestep sample
        :param noise_pred: model noise prediction
        :param t: current timestep we are at
        :return:
        """
        x0 = ((xt - (self.sqrt_one_minus_alpha_cum_prod.to(xt.device)[t] * noise_pred)) /
              torch.sqrt(self.alpha_cum_prod.to(xt.device)[t]))
        x0 = torch.clamp(x0, -1., 1.)
        
        mean = xt - ((self.betas.to(xt.device)[t]) * noise_pred) / (self.sqrt_one_minus_alpha_cum_prod.to(xt.device)[t])
        mean = mean / torch.sqrt(self.alphas.to(xt.device)[t])
        
        if t == 0:
            return mean, x0
        else:
            variance = (1 - self.alpha_cum_prod.to(xt.device)[t - 1]) / (1.0 - self.alpha_cum_prod.to(xt.device)[t])
            variance = variance * self.betas.to(xt.device)[t]
            sigma = variance ** 0.5
            z = torch.randn(xt.shape).to(xt.device)
            
            # OR
            # variance = self.betas[t]
            # sigma = variance ** 0.5
            # z = torch.randn(xt.shape).to(xt.device)
            return mean + sigma * z, x0
        
        
        
########## Create the noise scheduler #############
scheduler = LinearNoiseScheduler(
    num_timesteps=num_timesteps,
    beta_start=beta_start,
    beta_end=beta_end,
)
###############################################

In [9]:

# Instantiate the unet model_cond
model_cond = UnetCond(im_channels=z_channels).to(device)
model_cond.train()

vae = None
# Load VAE ONLY if latents are not to be saved or some are missing
print("Loading vqvae model_cond as latents not present")
vae = VQVAE(
    im_channels=im_channels,
).to(device)
vae.eval()

vae_ckpt_path = os.path.join(task_name, vqvae_autoencoder_ckpt_name + ".pt")
if os.path.exists(vae_ckpt_path):
    print("Loaded vae checkpoint")
    vae.load_state_dict(
        torch.load(
            vae_ckpt_path,
            map_location=device,
        )
    )
else:
    raise Exception("VAE checkpoint not found and use_latents was disabled")

# Specify training parameters
num_epochs = ldm_epochs
optimizer = Adam(model_cond.parameters(), lr=ldm_lr)
criterion = torch.nn.MSELoss()

# Load vae and freeze parameters ONLY if latents already not saved
if not use_latents:
    assert vae is not None
    for param in vae.parameters():
        param.requires_grad = False

Loading vqvae model_cond as latents not present
Loaded vae checkpoint


/tmp/ipykernel_1428973/566997539.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [10]:

def drop_class_condition(class_condition, class_drop_prob, im):
    if class_drop_prob > 0:
        class_drop_mask = torch.zeros((im.shape[0], 1), device=im.device).float().uniform_(0,
                                                                                           1) > class_drop_prob
        return class_condition * class_drop_mask
    else:
        return class_condition

In [ ]:
# Run training
for epoch_idx in range(num_epochs):
    losses = []
    for data in tqdm(loader):
        cond_input = None
        if use_condition:
            im, cond_input = data
        else:
            im = data
        optimizer.zero_grad()
        im = im.float().to(device)
        # Encode to latents only if the dataset is serving raw images
        # (dataset.use_latents is False when precomputed latents were not found on disk).
        if not dataset.use_latents:
            with torch.no_grad():
                im, _ = vae.encode(im)

        if "class" in condition_types and cond_input is not None:
            assert "class" in cond_input, (
                "Conditioning Type Class but no class conditioning input present"
            )
            class_condition = torch.nn.functional.one_hot(
                cond_input["class"],
                num_classes,
            ).to(device)
            # Use class_cond_drop_prob variable for drop probability
            class_drop_prob = class_cond_drop_prob
            # Drop condition
            cond_input["class"] = drop_class_condition(
                class_condition, class_drop_prob, im
            )
        ################################################

        # Sample random noise
        noise = torch.randn_like(im).to(device)

        # Sample timestep
        t = torch.randint(0, num_timesteps, (im.shape[0],)).to(device)

        # Add noise to images according to timestep
        noisy_im = scheduler.add_noise(im, noise, t)
        noise_pred = model_cond(noisy_im, t, cond_input=cond_input)
        loss = criterion(noise_pred, noise)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
    print("Finished epoch:{} | Loss : {:.4f}".format(epoch_idx + 1, np.mean(losses)))
    torch.save(
        model_cond.state_dict(),
        os.path.join(task_name, str(ldm_ckpt_name)),
    )

print("Done Training ...")


In [12]:
# import torch

# # Clear CUDA memory to help avoid OOM
# torch.cuda.empty_cache()
# torch.cuda.synchronize()

# Inference

In [13]:
# Instantiate the unet model_cond
model_cond = UnetCond(im_channels=z_channels).to(device)
model_cond.train()
model_cond_ckpt_path = os.path.join(task_name, ldm_ckpt_name)

model_cond.load_state_dict(
    torch.load(
        model_cond_ckpt_path,
        
        map_location=device,
    )
)


vae = None
# Load VAE ONLY if latents are not to be saved or some are missing
print("Loading vqvae model_cond as latents not present")
vae = VQVAE(
    im_channels=im_channels,
).to(device)
vae.eval()

vae_ckpt_path = os.path.join(task_name, vqvae_autoencoder_ckpt_name + ".pt")
if os.path.exists(vae_ckpt_path):
    print("Loaded vae checkpoint")
    vae.load_state_dict(
        torch.load(
            vae_ckpt_path,
            map_location=device,
        )
    )
else:
    raise Exception("VAE checkpoint not found and use_latents was disabled")

# Specify training parameters
num_epochs = ldm_epochs
optimizer = Adam(model_cond.parameters(), lr=ldm_lr)
criterion = torch.nn.MSELoss()

# Load vae and freeze parameters ONLY if latents already not saved
if not use_latents:
    assert vae is not None
    for param in vae.parameters():
        param.requires_grad = False

/tmp/ipykernel_1428973/1329705478.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Loading vqvae model_cond as latents not present
Loaded vae checkpoint


/tmp/ipykernel_1428973/1329705478.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [14]:
num_model_classes = model_cond.num_classes 
num_model_classes

10

In [16]:
from torchvision.utils import make_grid

In [ ]:
def sample_images(model, scheduler, vae, num_sample_classes=2):
    global task_name, ldm_ckpt_name, num_timesteps, z_channels, im_size

    num_model_classes = model.num_classes        # source of truth

    sample_classes = torch.arange(num_sample_classes, device=device)
    print('Generating images for classes:', sample_classes.tolist())

    xt = torch.randn(
        (len(sample_classes), z_channels, im_size, im_size), device=device
    )

    cond_input = {
        'class': torch.nn.functional.one_hot(
            sample_classes, num_classes=num_model_classes
        ).float().to(device)
    }
    uncond_input = {'class': torch.zeros_like(cond_input['class'])}

    cf_guidance_scale = 1.0

    samples_dir = os.path.join(task_name, 'cond_class_samples')
    os.makedirs(samples_dir, exist_ok=True)

    for i in tqdm(range(num_timesteps - 1, -1, -1), total=num_timesteps):
        t = torch.full((xt.shape[0],), i, device=device, dtype=torch.long)

        noise_pred_cond = model(xt, t, cond_input)
        if cf_guidance_scale > 1:
            noise_pred_uncond = model(xt, t, uncond_input)
            noise_pred = noise_pred_uncond + cf_guidance_scale * (noise_pred_cond - noise_pred_uncond)
        else:
            noise_pred = noise_pred_cond

        xt, x0_pred = scheduler.sample_prev_timestep(
            xt, noise_pred, torch.tensor(i, device=device)
        )

        ims = vae.decode(xt) if i == 0 else x0_pred
        ims = torch.clamp(ims, -1., 1.).detach().cpu()
        ims = (ims + 1) / 2

        grid = make_grid(ims, nrow=1)
        img = torchvision.transforms.ToPILImage()(grid)
        img.save(os.path.join(samples_dir, f'x0_{i}.png'))
        img.close()

    print(f"Image samples saved in {samples_dir}")


sample_images(model_cond, scheduler, vae, num_sample_classes=2)

In [18]:
import glob
from PIL import Image

# Directory containing images
samples_dir = "vqvae_train/cond_class_samples"

# Get all png files and sort them by timestep in filename
image_files = sorted(
    glob.glob(os.path.join(samples_dir, "x0_*.png")),
    key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("_")[1])
)

# Load images
images = [Image.open(f) for f in image_files]

# Save as GIF
gif_path = os.path.join(samples_dir, "generation.gif")
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=100,
    loop=0
)

print(f"GIF saved to {gif_path}")

GIF saved to vqvae_train/cond_class_samples/generation.gif
